# 06 — Exploration de la génération : chunks -> réponse citée par Claude

**Objectif** : voir concrètement ce que renvoie Claude Sonnet 5 avec les chunks passés en blocs `search_result`, avant d'écrire `rag/generation.py`.

1. connexion, moteur de recherche et client Anthropic ;
2. le prompt système v1, lu depuis le package ;
3. des résultats de recherche aux blocs `search_result` (deux granularités) ;
4. l'appel à l'API et la lecture de la réponse : blocs de réflexion, texte, citations ;
5. une première question, affichée avec ses notes de citation ;
6. l'abstention sur une question hors corpus ;
7. effort et réflexion : latence, tokens, qualité ;
8. granularité des citations : chunk entier ou paragraphes ;
9. part de la réponse couverte par une citation (un premier indicateur d'ancrage).

Prérequis : `ANTHROPIC_API_KEY` dans le `.env`, `docker compose up -d`, base indexée.

> **Coût** : chaque appel envoie environ 5 chunks et coûte de l'ordre du centime. La section 7 fait 4 appels par question. Les sorties contiennent du texte du livre (passages cités) : elles restent en local, nbstripout les vide au commit.

## 1. Connexion, recherche et client

On réutilise le code propre : `MoteurRecherche` (avec son garde-fou de modèle) et la config unique `rag.yaml`. Le client Anthropic lit `ANTHROPIC_API_KEY` dans l'environnement, rempli par `load_dotenv`.

In [1]:
import time
from importlib import resources

import anthropic
import psycopg
from dotenv import load_dotenv

from assistant_regles.ingest.config import trouver_racine
from assistant_regles.rag.config import charger_config
from assistant_regles.rag.embeddings import EncodeurBGEM3
from assistant_regles.rag.recherche import MoteurRecherche

load_dotenv(trouver_racine() / ".env")
config = charger_config()

conn = psycopg.connect(autocommit=True)
moteur = MoteurRecherche(EncodeurBGEM3.charger(config.embeddings), conn)
client = anthropic.Anthropic()

MODELE = "claude-sonnet-5"
print("anthropic", anthropic.__version__, "| modèle :", MODELE)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

anthropic 1.8.0 | modèle : claude-sonnet-5


## 2. Le prompt système v1

Lu depuis le package avec `importlib.resources`, comme le fera `generation.py`. 
On peut le modifier dans `src/assistant_regles/rag/prompts/systeme_v1.md` puis relancer cette cellule.

In [2]:
PROMPT_SYSTEME = (
    resources.files("assistant_regles.rag").joinpath("prompts/systeme_v1.md").read_text(encoding="utf-8")
)
print(PROMPT_SYSTEME)

Tu es un assistant d'arbitrage pour le jeu de figurines Warhammer 40,000, 11e édition.
Tu réponds aux questions des joueurs sur les règles de base, à partir d'extraits
du livre de règles fournis avec chaque question.

Règles impératives :
1. Réponds uniquement à partir des extraits fournis. N'utilise pas tes connaissances
   préalables du jeu : elles peuvent venir d'éditions antérieures, dont les règles
   diffèrent, et une réponse plausible mais issue d'une autre édition induirait
   le joueur en erreur.
2. Si les extraits ne permettent pas de répondre, commence ta réponse exactement par :
   « Je ne trouve pas la réponse dans les extraits du livre de règles fournis. »
   Tu peux ensuite indiquer ce qui manque. N'invente jamais de règle.
3. Si les extraits ne répondent qu'en partie, donne la partie qu'ils étayent et
   signale clairement ce qui n'est pas couvert.
4. Si deux extraits semblent se contredire, signale-le au lieu de trancher.

Style :
- Réponds en français, de façon claire

## 3. Des résultats de recherche aux blocs `search_result`

Chaque chunk devient un bloc :

- `source` : un identifiant stable (ici l'id du chunk) ;
- `title` : ce qu'un humain lira dans la citation (code, sous-section, pages) ;
- `content` : une liste de blocs de texte. **Le bloc de texte est la plus petite unité citable** :
  - granularité `"chunk"` : un seul bloc, donc chaque citation renvoie le chunk entier ;
  - granularité `"paragraphe"` : un bloc par paragraphe (séparés par une ligne vide), donc des citations plus précises.
- `citations: {"enabled": True}` : obligatoire sur **tous** les blocs de la requête, ou sur aucun.

La question vient **après** les extraits dans le message utilisateur : c'est l'ordre recommandé quand le contexte est long.

In [4]:
def titre(r):
    """Titre lisible d'un résultat : code, sous-section, pages."""
    pages = f"p. {r.page_debut}" + (f"-{r.page_fin}" if r.page_fin != r.page_debut else "")
    return f"{r.code or '(sans code)'} {r.sous_section or r.section_titre or ''} — {pages}".strip()


def vers_search_result(r, granularite="chunk"):
    """Convertit un Resultat de la recherche en bloc search_result de l'API."""
    if granularite == "paragraphe":
        morceaux = [p.strip() for p in r.texte.split("\n\n") if p.strip()]
    else:
        morceaux = [r.texte]
    return {
        "type": "search_result",
        "source": f"chunk:{r.id}",
        "title": titre(r),
        "content": [{"type": "text", "text": m} for m in morceaux],
        "citations": {"enabled": True},
    }


def construire_message(question, resultats, granularite="chunk"):
    """Message utilisateur : les extraits d'abord, la question ensuite."""
    return {
        "role": "user",
        "content": [vers_search_result(r, granularite) for r in resultats]
        + [{"type": "text", "text": question}],
    }


# Aperçu sur une question (textes tronqués pour la lisibilité)
resultats = moteur.rechercher("Les pistolets peuvent-ils être utilisés au corps à corps ?", k=5)
for bloc in construire_message("…", resultats, "paragraphe")["content"][:2]:
    print(bloc["title"], "|", bloc["source"], "|", len(bloc["content"]), "bloc(s) de texte")

24.27 [PISTOLET] — p. 84 | chunk:ed379645-8f3c-5ce8-9ec7-eddadf8166c2 | 2 bloc(s) de texte
24.07 [COMBAT RAPPROCHÉ] — p. 81 | chunk:a0ac11e3-77f4-5ce2-9768-976265d1d0ab | 2 bloc(s) de texte


## 4. Appel à l'API et lecture de la réponse

Paramètres :

- **pas de `temperature`** : sur Sonnet 5, une valeur non par défaut renvoie une erreur 400 ;
- `thinking` : absent = réflexion adaptative (défaut de Sonnet 5) ; `{"type": "disabled"}` = pas de réflexion ;
- `output_config={"effort": ...}` : `low`, `medium` ou `high` (défaut) ;
- `max_tokens` borne **toute** la sortie, réflexion comprise.

Si ta version du SDK refuse `output_config` (`unexpected keyword argument`), mets-le à jour : `uv lock --upgrade-package anthropic && uv sync`.

La réponse est une **liste de blocs** :

- `thinking` : la réflexion (résumée), à ne pas montrer à l'utilisateur ;
- `text` : des segments de texte. Ceux qui s'appuient sur un extrait portent une liste `citations`, dont chaque élément donne `search_result_index` (position du bloc cité dans la requête), `cited_text` (le texte exact cité) et `title`.

`afficher` recompose la réponse avec des notes numérotées [1], [2]… puis liste les sources.

In [5]:
def appeler(question, k=5, effort="medium", reflexion=True, granularite="chunk", max_tokens=4096):
    """Recherche puis génération ; renvoie (réponse API, résultats de recherche, durée en s)."""
    resultats = moteur.rechercher(question, k=k)
    parametres = {
        "model": MODELE,
        "max_tokens": max_tokens,
        "system": PROMPT_SYSTEME,
        "messages": [construire_message(question, resultats, granularite)],
        "output_config": {"effort": effort},
    }
    if not reflexion:
        parametres["thinking"] = {"type": "disabled"}
    debut = time.perf_counter()
    reponse = client.messages.create(**parametres)
    return reponse, resultats, time.perf_counter() - debut


def afficher(question, reponse, resultats, duree=None, montrer_reflexion=False):
    """Réponse avec notes de citation numérotées, puis la liste des sources."""
    notes = {}  # (index du résultat, début, fin) -> numéro de note
    morceaux = []
    for bloc in reponse.content:
        if bloc.type == "thinking" and montrer_reflexion:
            print("[réflexion]", bloc.thinking[:500], "…\n")
        if bloc.type != "text":
            continue
        morceaux.append(bloc.text)
        for c in bloc.citations or []:
            cle = (c.search_result_index, c.start_block_index, c.end_block_index)
            notes.setdefault(cle, len(notes) + 1)
            morceaux.append(f" [{notes[cle]}]")

    print("=" * 100)
    print("QUESTION :", question)
    print("=" * 100)
    print("".join(morceaux))
    print("\n--- sources ---")
    for (i, debut, fin), numero in notes.items():
        cite = next(c for b in reponse.content if b.type == "text"
                    for c in (b.citations or []) if c.search_result_index == i
                    and c.start_block_index == debut and c.end_block_index == fin)
        extrait = " ".join(cite.cited_text.split())
        print(f"[{numero}] {cite.title}  (résultat #{i + 1} de la recherche, blocs {debut}-{fin})")
        print(f"     « {extrait[:160]}{' …' if len(extrait) > 160 else ''} »")
    u = reponse.usage
    print(f"\nstop_reason={reponse.stop_reason} | tokens entrée={u.input_tokens} sortie={u.output_tokens}"
          + (f" | {duree:.1f} s" if duree is not None else ""))

## 5. Une première question

Lis la réponse avec ton regard de joueur :

- est-elle **juste** ?
- chaque affirmation importante porte-t-elle une note ?
- les sources citées sont-elles les bons chunks ?

`montrer_reflexion=True` affiche le début de la réflexion, instructif pour comprendre comment Claude a utilisé les extraits.

In [6]:
question = "Les pistolets peuvent-ils être utilisés au corps à corps ?"
reponse, resultats, duree = appeler(question, effort="medium")
afficher(question, reponse, resultats, duree, montrer_reflexion=True)

QUESTION : Les pistolets peuvent-ils être utilisés au corps à corps ?
Oui. Les pistolets peuvent être utilisés pour tirer à bout portant, et [PISTOLET] et [COMBAT RAPPROCHÉ] sont identiques au regard des règles. [1]

Concrètement, cela renvoie à la règle [COMBAT RAPPROCHÉ] : les unités contenant une ou plusieurs figurines avec une arme de [COMBAT RAPPROCHÉ] peuvent tirer en utilisant le tir en combat rapproché (10.06). Quand vous utilisez un autre type de tir, pour chaque figurine de cette unité (sauf les figurines de MONSTRE/VÉHICULE), vous pouvez choisir une seule des options suivantes avec laquelle effectuer des attaques : une ou plusieurs de ses armes de [COMBAT RAPPROCHÉ], ou une ou plusieurs de ses autres armes de tir. [2]

À noter également : [PISTOLET] est une aptitude qui existait précédemment et qui sera supplantée par [COMBAT RAPPROCHÉ] au fur et à mesure de l'enrichissement de la présente édition de Warhammer 40,000. Les deux sont fonctionnellement les mêmes, mais le nom de

## 6. Abstention : une question hors corpus

La recherche renvoie quand même 5 chunks (sans rapport). Le prompt impose une phrase d'abstention fixe : on vérifie qu'elle apparaît **au début** de la réponse, ce que l'évaluation contrôlera automatiquement.

Essaie aussi une question **plausible mais absente** du livre de base (une règle d'un codex, par exemple). C'est le cas le plus dangereux : Claude connaît peut-être la réponse d'une autre source, et doit quand même s'abstenir.

In [8]:
PHRASE_ABSTENTION = "Je ne trouve pas la réponse dans les extraits du livre de règles fournis."

for question in [
    "Quelle est la recette de la pâte à crêpes ?",
    "Quelle est la règle de faction des nécrons?",
    "Quel personnage possède une règle pour ajouter des CP gratuit chaque tour?"
]:
    reponse, resultats, duree = appeler(question, effort="medium")
    afficher(question, reponse, resultats, duree)
    texte = "".join(b.text for b in reponse.content if b.type == "text").strip()
    print("-> abstention détectée :", texte.startswith(PHRASE_ABSTENTION), "\n")

QUESTION : Quelle est la recette de la pâte à crêpes ?
Je ne trouve pas la réponse dans les extraits du livre de règles fournis. Les extraits fournis concernent les règles de Warhammer 40,000 (dés, jets de touche, aptitudes de base, stratagèmes, etc.) et ne contiennent aucune information relative à une recette de cuisine, comme celle de la pâte à crêpes.

--- sources ---

stop_reason=end_turn | tokens entrée=3694 sortie=123 | 2.4 s
-> abstention détectée : True 

QUESTION : Quelle est la règle de faction des nécrons?
Je ne trouve pas la réponse dans les extraits du livre de règles fournis. Les extraits disponibles expliquent uniquement le principe général des aptitudes de faction : certaines aptitudes sont communes à chaque unité appartenant à une faction particulière, appelées aptitudes de faction ou règles d'armée, listées dans la section Aptitudes de Faction d'une fiche technique, et elles s'appliquent seulement si la faction d'armée choisie correspond à un mot-clé de faction listé 

## 7. Effort et réflexion : latence, tokens, qualité

Même question, quatre réglages. À comparer :

- la **latence**, déterminante pour une interface de chat ;
- les **tokens de sortie**, qui incluent la réflexion (et donc le coût) ;
- la **qualité** de la réponse : c'est toi qui juges.

Si `low` ou « sans réflexion » donnent des réponses aussi bonnes sur des questions de règles, c'est un gain net pour l'interface. Mets une question **difficile** (règles proches, multi-sections) : c'est là qu'un effort élevé peut se justifier.

In [9]:
question = "Explique moi comment se déroule une phase de charge précisément."  # à adapter pour tester l'effort nécessaire

reglages = [
    ("sans réflexion", {"effort": "low", "reflexion": False}),
    ("effort low", {"effort": "low"}),
    ("effort medium", {"effort": "medium"}),
    ("effort high", {"effort": "high"}),
]
comparaison = []
for nom, params in reglages:
    reponse, resultats, duree = appeler(question, **params)
    texte = "".join(b.text for b in reponse.content if b.type == "text")
    reflexion = any(b.type == "thinking" for b in reponse.content)
    comparaison.append((nom, duree, reponse.usage.output_tokens, reflexion, texte))

print(f"{'réglage':>15}  {'durée':>6}  {'sortie':>7}  réflexion")
for nom, duree, sortie, reflexion, _ in comparaison:
    print(f"{nom:>15}  {duree:5.1f}s  {sortie:7d}  {reflexion}")
for nom, *_, texte in comparaison:
    print(f"\n----- {nom} -----\n{texte}")

        réglage   durée   sortie  réflexion
 sans réflexion   11.4s     1382  False
     effort low   11.2s     1447  False
  effort medium   12.3s     1604  False
    effort high   11.7s     1502  False

----- sans réflexion -----
La phase de charge se déroule en trois étapes, résolues dans l'ordre : Début de la phase de charge, Charger, Fin de la phase de charge.

## 11.01 Début de la phase de charge
Les règles déclenchées au début de la phase de charge sont résolues maintenant.

## 11.02 Charger
Le joueur actif résout des charges avec ses unités éligibles, une à la fois, en utilisant la séquence ci-dessous, jusqu'à ce que toutes les unités avec lesquelles il a choisi de charger aient déclaré une charge et que ces charges aient été résolues.

Cette séquence comporte trois sous-étapes :

**1. Déclarez une Charge** : Choisissez une unité amie qui n'a pas déclaré de charge à cette phase et qui est éligible pour déclarer une charge. Cette unité déclare une charge.

Une unité n'est **pas*

## 8. Granularité des citations : chunk entier ou paragraphes

Avec un seul bloc par chunk, chaque citation renvoie tout le chunk (jusqu'à 600 tokens) : l'utilisateur doit chercher le passage utile lui-même. Avec un bloc par paragraphe, la citation pointe le paragraphe précis.

À observer : la longueur moyenne du texte cité, et si les citations « paragraphe » tombent bien sur la phrase qui justifie l'affirmation.

In [10]:
question = "Les pistolets peuvent-ils être utilisés au corps à corps ?"  # à adapter pour cibler une règle précise uniquement ici

for granularite in ("chunk", "paragraphe"):
    reponse, resultats, duree = appeler(question, effort="medium", granularite=granularite)
    citations = [c for b in reponse.content if b.type == "text" for c in (b.citations or [])]
    longueurs = [len(c.cited_text) for c in citations]
    moyenne = sum(longueurs) / len(longueurs) if longueurs else 0
    print(f"\n##### granularité {granularite} : {len(citations)} citation(s), "
          f"texte cité moyen {moyenne:.0f} caractères")
    afficher(question, reponse, resultats, duree)


##### granularité chunk : 3 citation(s), texte cité moyen 554 caractères
QUESTION : Les pistolets peuvent-ils être utilisés au corps à corps ?
Oui. [PISTOLET] et [COMBAT RAPPROCHÉ] sont identiques au regard des règles. [1]

Cela signifie que les armes dotées du mot-clé [PISTOLET] fonctionnent exactement comme des armes de [COMBAT RAPPROCHÉ]. Concrètement, cela implique que : les unités contenant une ou plusieurs figurines avec une arme de [COMBAT RAPPROCHÉ] peuvent tirer en utilisant le tir en combat rapproché (10.06). [2]

À noter également, comme précisé dans une note de conception : [PISTOLET] est une aptitude qui existait précédemment et qui sera supplantée par [COMBAT RAPPROCHÉ] au fur et à mesure de l'enrichissement de la présente édition de Warhammer 40,000. Les deux sont fonctionnellement les mêmes, mais le nom de la seconde sied mieux aux armes qui fonctionnent de cette façon. [1]

En résumé : un pistolet ([PISTOLET]) fonctionne comme une arme de [COMBAT RAPPROCHÉ], ce qui pe

pas mal, on constate que le renvoi du paragraphe permet aussi de cibler certains passages supplémentaires qui apportent des précisions importante sur la règle. Cela permet de bien compléter la réponse du joueur car celui-ci pose une question sans savoir qu'il peut exister des cas spécifiques. Le but de ce RAG est justement d'être aussi capable de renvoyer ces passages supplémentaires. 

Ici l'utilisation de paragraphe semble plus appropriée.

## 9. Part de la réponse couverte par une citation

Indicateur simple, calculé sur la dernière réponse : la proportion du texte (en caractères) portée par des segments **cités**. Les segments non cités sont souvent des transitions (« Oui. », « En revanche, ») ; mais une **affirmation de règle** sans citation est suspecte : Claude l'a peut-être tirée de sa mémoire.

Ce n'est pas une mesure de fidélité : RAGAS s'en chargera à l'évaluation.

In [11]:
segments = [b for b in reponse.content if b.type == "text"]
total = sum(len(b.text) for b in segments)
cite = sum(len(b.text) for b in segments if b.citations)
print(f"texte cité : {cite}/{total} caractères = {cite / total:.0%}" if total else "réponse vide")

print("\nSegments NON cités (à relire : transition ou affirmation non étayée ?) :")
for b in segments:
    if not b.citations and b.text.strip():
        print("  -", repr(b.text.strip()[:150]))

texte cité : 697/1621 caractères = 43%

Segments NON cités (à relire : transition ou affirmation non étayée ?) :
  - '**Oui**, mais pas exactement « au corps à corps » au sens des attaques de mêlée : le [PISTOLET] permet de tirer alors que la figurine est engagée.\n\nPr'
  - ". Il faut donc se référer à la règle [COMBAT RAPPROCHÉ] pour comprendre son fonctionnement complet.\n\nD'après cette règle,"
  - ". Autrement dit, une arme dotée de l'aptitude [PISTOLET]/[COMBAT RAPPROCHÉ] permet à sa figurine de tirer même si elle est engagée (ce qui est normale"
  - ".\n\nEn revanche, il est explicitement précisé qu'une figurine avec cette aptitude ne peut pas viser une cible avec une arme à [DÉFLAGRATION] via le tir"
  - ".\n\nEn résumé : le [PISTOLET] (identique à [COMBAT RAPPROCHÉ]) sert à tirer alors que la figurine est engagée en mêlée, ce n'est pas une arme de mêlée "


## Bilan : décisions pour `rag/generation.py`

- **effort** et **réflexion** retenus (section 7), selon la latence acceptable pour le chat ;
- **granularité** des citations (section 8) ;
- `max_tokens` : regarde `stop_reason`. `max_tokens` signifie une réponse tronquée ;
- l'**abstention** fonctionne-t-elle, y compris sur une question plausible mais hors livre (section 6) ?
- les **retouches du prompt** (qui deviendra `systeme_v2.md` si tu le modifies après ces essais) ;
- la **part citée** typique (section 9).

In [12]:
conn.close()

:)